In [1]:
import requests
import pandas as pd
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

url = "https://api.hsx.vn/mk/api/v1/market/trading-summary"

# ===== session + retry =====
session = requests.Session()

retries = Retry(
    total=5,
    backoff_factor=1,   # delay tăng dần
    status_forcelist=[500, 502, 503, 504]
)

adapter = HTTPAdapter(max_retries=retries)
session.mount("https://", adapter)

headers = {
    "Accept": "application/json, text/plain, */*",
    "Origin": "https://www.hsx.vn",
    "Referer": "https://www.hsx.vn/",
    "User-Agent": "Mozilla/5.0",
    "type": "HJ2HNS3SKICV4FNE"
}

# ===== dates =====
dates = pd.date_range(start="2023-01-31", end="2025-12-31", freq="6ME")
dates = dates.union(pd.to_datetime(["2025-12-31"]))

all_data = []

for d in dates:
    params = {
        "date": d.strftime("%Y-%m-%d"),
        "securitiesType": "S"
    }

    try:
        res = session.get(url, headers=headers, params=params, timeout=30)
        res.raise_for_status()

        json_data = res.json()
        all_data.extend(json_data.get("data", []))

        print(f"✅ Done {d.date()}")

        time.sleep(1)  # 👈 giảm tốc độ gọi

    except Exception as e:
        print(f"❌ Lỗi tại {d}: {e}")

# ===== dataframe =====
df = pd.DataFrame(all_data)

df = df.drop_duplicates(subset=["year", "month"])
df = df[(df["year"] >= 2023) & (df["year"] <= 2025)]
df = df.sort_values(by=["year", "month"])

# clean numeric
for col in ["mainVolume", "mainValue", "bigLotVolume", "bigLotValue"]:
    if col in df.columns:
        df[col] = df[col].str.replace(",", "").astype(float)

df["Month-Year"] = df["year"].astype(str) + "-" + df["month"].astype(str).str.zfill(2)

print(df)

df.to_csv("hsx_full_2023_2025_full_columns.csv", index=False)

✅ Done 2023-01-31
✅ Done 2023-07-31
✅ Done 2024-01-31
✅ Done 2024-07-31
✅ Done 2025-01-31
✅ Done 2025-07-31
✅ Done 2025-12-31
    month  year    mainVolume     mainValue  bigLotVolume  bigLotValue  \
0       1  2023  8.085991e+07  1.421605e+08   11960506.28  25803304.57   
17      2  2023  1.006213e+08  1.704392e+08   12917504.68  29851393.08   
16      3  2023  1.043953e+08  1.817149e+08   13944524.47  29060765.09   
15      4  2023  1.161601e+08  1.952730e+08   12087589.11  27153889.63   
14      5  2023  1.285760e+08  2.125921e+08   13658737.76  31527489.66   
13      6  2023  1.771461e+08  3.324227e+08   15906578.16  39145936.05   
12      7  2023  1.696170e+08  3.491707e+08   14242606.22  34490727.31   
29      8  2023  2.128436e+08  4.691207e+08   15998433.15  38524078.53   
28      9  2023  1.697524e+08  4.043156e+08   14122364.08  38353418.17   
27     10  2023  1.266505e+08  2.872661e+08   11045489.63  27009934.49   
26     11  2023  1.503511e+08  3.280260e+08   15834549.03  3

PermissionError: [Errno 13] Permission denied: 'hsx_full_2023_2025_full_columns.csv'